# Forecast TTE v0 — Exploración de Datos y Auditoría de Calidad

**Objetivo:** Construir desde cero el sistema de forecasting de Ausentismo (ABS) y Turnover (TO)
para TTE — Brasil, Shipping. No dependemos de tablas DW de terceros.

**Enfoque:** Antes de modelar, entendemos los datos a fondo.

---

## Roadmap

| Celda | Propósito |
|---|---|
| 1 | Imports y configuración global |
| 2 | Conexión y test BigQuery |
| 3 | Schema de la tabla madre ABS + muestra cruda |
| 4 | Exploración volumétrica: países, job classifications, sitios |
| 5 | Exploración de la tabla madre TO |
| 6 | Nuestra query de agregación ABS (directa desde Silver) |
| 7 | Auditoría de calidad: nulos, imposibles, duplicados, gaps |
| 8 | Análisis de distribuciones por variable |
| 9 | Detección de outliers (IQR + Z-score modificado + visual) |
| 10 | Análisis de series temporales por sitio (heatmap, lifecycles) |

---

> **Tablas fuente (solo lectura):**  
> `meli-people.SILVER_PE_SHIPPING.LK_PE_SHIPPING_ABSENCES_PP`  
> `meli-people.SILVER_PE_SHIPPING.KPI_LATAM_NC_TO_ALL`  
> `meli-people.STG_PE_SHIPPING.ONLY_TTE_REG_CLASSIFICADOR`


---
## Celda 1 — Imports y Configuración Global

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from google.cloud import bigquery

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ID   = 'meli-people'
PAIS         = 'Brasil'
FECHA_INICIO = '2024-01-01'

TBL_ABS = 'meli-people.SILVER_PE_SHIPPING.LK_PE_SHIPPING_ABSENCES_PP'
TBL_TO  = 'meli-people.SILVER_PE_SHIPPING.KPI_LATAM_NC_TO_ALL'
TBL_REG = 'meli-people.STG_PE_SHIPPING.ONLY_TTE_REG_CLASSIFICADOR'
JOB_SQL = "'Rep de Envio 1', 'Rep de Envio 2'"

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('OK Imports')
print(f'   Proyecto : {PROJECT_ID}')
print(f'   Pais     : {PAIS}')
print(f'   Desde    : {FECHA_INICIO}')
print(f'   Cache dir: {DATA_DIR.resolve()}')


OK Imports
   Proyecto : meli-people
   Pais     : Brasil
   Desde    : 2024-01-01
   Cache dir: C:\Users\sergibarra\Documents\SI_People_Analytics_2026\Forecast_TTE\Forecast_TTE_v0\data


---
## Celda 2 — Conexión y Test BigQuery

En Vertex/Colab ya está autenticado.
En local ejecutar `gcloud auth application-default login` una sola vez.


In [2]:
client = bigquery.Client(project=PROJECT_ID)

# Helper para todas las queries del proyecto.
# create_bqstorage_client=False: usa HTTP en vez de gRPC
# (BigQuery Storage API requiere permisos extra que no tenemos)
def bq_query(sql: str):
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

# Test de conexion
_n = bq_query(f'SELECT COUNT(*) AS n FROM `{TBL_ABS}`').iloc[0, 0]
print(f'OK Conexion BigQuery - {PROJECT_ID}')
print(f'   {TBL_ABS}')
print(f'   Total filas tabla madre ABS: {_n:,}')


OK Conexion BigQuery - meli-people
   meli-people.SILVER_PE_SHIPPING.LK_PE_SHIPPING_ABSENCES_PP
   Total filas tabla madre ABS: 70,254,900


---
## Celda 3 — Schema de la Tabla Madre ABS

Antes de filtrar nada, vemos todas las columnas disponibles y una muestra cruda.
Esto evita asumir cosas sobre la data y puede revelar columnas útiles que se ignoraban.


In [3]:
schema_q = """
    SELECT column_name, data_type, is_nullable
    FROM `meli-people.SILVER_PE_SHIPPING.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = 'LK_PE_SHIPPING_ABSENCES_PP'
    ORDER BY ordinal_position
"""
df_schema_abs = bq_query(schema_q)
print(f'Columnas tabla madre ABS: {len(df_schema_abs)}')
display(df_schema_abs)


Columnas tabla madre ABS: 41


,column_name,data_type,is_nullable
0,YEAR_MONTH_DAY,DATE,YES
1,YEAR,INT64,YES
2,MONTH,INT64,YES
3,WEEK_DATE,DATE,YES
4,WEEK,STRING,YES
5,USER_ID,STRING,YES
6,USER_FILE_ID,STRING,YES
7,COUNTRY,STRING,YES
8,LOCATION_COD,STRING,YES
9,SITE,STRING,YES


In [4]:
sample_q = f"""
    SELECT * FROM `{TBL_ABS}`
    WHERE COUNTRY = '{PAIS}'
      AND YEAR_MONTH_DAY >= DATE '{FECHA_INICIO}'
    LIMIT 50
"""
df_sample = bq_query(sample_q)

print(f'Muestra: {len(df_sample)} filas x {len(df_sample.columns)} cols')
display(df_sample.head(20))


Muestra: 50 filas x 41 cols


,YEAR_MONTH_DAY,YEAR,MONTH,WEEK_DATE,WEEK,USER_ID,USER_FILE_ID,COUNTRY,LOCATION_COD,SITE,LOCATION_TYPE,AREA,SUBAREA,JOB_CLASSIFICATION,EXTERNAL_CONTRACTOR,QUALIFIER_CODE,ABSENCE_TYPE,REASON_ABSENCE,SYSTEM_ORIGIN,PAYROLL,DOTACION,ABSENCE_MANAGEABLE,ABSENCE_NOT_MANAGEABLE,ABSENCE_OTHERS,TOTAL_EXTERNAL_ABSENCE,TOTAL_INTERNAL_ABSENCE,ABSENCE_MONDAY,ABSENCE_TUESDAY,ABSENCE_WEDNESDAY,ABSENCE_THURSDAY,ABSENCE_FRIDAY,ABSENCE_SATURDAY,ABSENCE_SUNDAY,PAYMENT_CLASSIFICATION,ABSENCE_PAYMENT,ABSENCE_NO_PAYMENT,INSERT_DATE,AUD_INS_DTTM,AUD_UPD_DTTM,ABSENCE_OPERATIONAL_TYPE,ABSENCE_NR
0,2025-10-22,2025,10,2025-10-25,Week43,30003559,14183,Brasil,0122_EBA,MELICIDADE,Oficina,Logistics Full Inbound & Removals,Logistics Full Inbound & Removals,Asistente,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-12-22,2026-04-30,2026-04-30,DOTACION,0.0000
1,2025-10-22,2025,10,2025-10-25,Week43,30048917,310933,Brasil,3785_EBA,FBM - FRANCO DE ROCHA BRSP10,FBM,Fulfillment Brazil,Operations Execution - Outbound,Rep de Envio 1,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-12-22,2026-04-30,2026-04-30,DOTACION,0.0000
2,2025-10-22,2025,10,2025-10-25,Week43,30052365,312632,Brasil,CDCJ_EBA,FBM - CAJAMAR BRSP02,FBM Ext,ICQA,ICQA Execution,Operador Logístico 1,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-12-22,2026-04-30,2026-04-30,DOTACION,0.0000
3,2025-10-22,2025,10,2025-10-25,Week43,30040571,306875,Brasil,SP21_EBA,FBM - CAJAMAR BRSP14,FBM,Fulfillment Brazil,Operations Execution - Inbound,Sr Team Leader - Shipping,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-12-22,2026-04-30,2026-04-30,DOTACION,0.0000
4,2025-10-22,2025,10,2025-10-25,Week43,30040860,306953,Brasil,XSPI_EBA,EXT - SÃO PAULO INTERIOR,Oficina Remoto,Plant Engineering Execution,Plant Engineering Execution UTR - Brazil,Analista Semi Senior,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-12-22,2026-04-30,2026-04-30,DOTACION,0.0000
5,2025-10-22,2025,10,2025-10-25,Week43,30030980,301576,Brasil,SP06_EBA,FBM - ARAÇARIGUAMA BRSP06,FBM,Fulfillment Special Operations,Operations Execution - Outbound,Rep de Envio 2,No,Inss (Doença),No Gestionable,Art/ Accidentes De Trabajo,AHGORA,No,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0,0,1,0,0,0,0,NO PAGO,0,1,2025-12-22,2026-04-30,2026-04-30,NO OPERACIONAL,1.0000
6,2025-06-29,2025,6,2025-07-05,Week27,BR2097323,2097323,Brasil,N/A,FBM - ARAÇARIGUAMA BRSP06,FBM,N/A,N/A,Rep de Envio 1,Si,DE - Desligado,N/A,N/A,FLOW - MLB,No,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,N/A,0,0,2026-04-24,2026-04-30,2026-04-30,N/A,0.0000
7,2025-06-29,2025,6,2025-07-05,Week27,BR2117493,2117493,Brasil,N/A,FBM - ARAÇARIGUAMA BRSP06,FBM,N/A,N/A,Rep de Envio 1,Si,DSR - Escala,N/A,N/A,FLOW - MLB,No,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,N/A,0,0,2026-04-24,2026-04-30,2026-04-30,N/A,0.0000
8,2025-06-29,2025,6,2025-07-05,Week27,BR1976062,1976062,Brasil,N/A,FBM - ARAÇARIGUAMA BRSP06,FBM,N/A,N/A,Rep de Envio 1,Si,DSR - Escala,N/A,N/A,FLOW - MLB,No,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,N/A,0,0,2026-04-24,2026-04-30,2026-04-30,N/A,0.0000
9,2025-06-29,2025,6,2025-07-05,Week27,30223100,372820,Brasil,SP06_EBA,FBM - ARAÇARIGUAMA BRSP06,FBM,Fulfillment Special Operations,Operations Execution - Outbound,Rep de Envio 1,No,Horas Previstas,Dotacion,Dotacion,AHGORA,No,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0,0,0,0,0,0,0,DOTACION,0,0,2025-08-28,2026-04-30,2026-04-30,DOTACION,0.0000


---
## Celda 4 — Exploración Volumétrica de la Tabla Madre ABS

Entendemos el universo completo **antes** de aplicar filtros de negocio.

**Preguntas clave:**
- ¿Cuántos registros por país, año, tipo de ausencia?
- ¿Qué `JOB_CLASSIFICATION` existen? ¿El filtro actual captura todo lo relevante?
- ¿Cuántos sitios únicos en Brasil?
- ¿Qué % son MELI vs Externos?


In [5]:
# Distribución por País
df_vol_pais = bq_query(f"""
    SELECT COUNTRY,
        COUNT(*)                                                AS registros,
        COUNT(DISTINCT SITE)                                    AS sitios_unicos,
        MIN(YEAR_MONTH_DAY)                                     AS fecha_min,
        MAX(YEAR_MONTH_DAY)                                     AS fecha_max,
        COUNTIF(ABSENCE_TYPE = 'Dotacion')                      AS reg_dotacion,
        COUNTIF(ABSENCE_TYPE = 'Gestionable')                   AS reg_gestionable,
        ROUND(COUNTIF(EXTERNAL_CONTRACTOR='Si')/COUNT(*)*100,2) AS pct_externos
    FROM `{TBL_ABS}`
    WHERE YEAR_MONTH_DAY >= DATE '{FECHA_INICIO}'
    GROUP BY COUNTRY ORDER BY registros DESC
""")
print('Distribucion por Pais - tabla madre ABS')
display(df_vol_pais)


Distribucion por Pais - tabla madre ABS


,COUNTRY,registros,sitios_unicos,fecha_min,fecha_max,reg_dotacion,reg_gestionable,pct_externos
0,Brasil,34976818,449,2024-01-01,2026-08-12,27138777,2151894,13.6200
1,México,20338045,286,2024-01-01,2026-08-12,18719887,921088,9.6900
2,Argentina,2365932,15,2024-01-01,2026-08-12,2186645,69430,11.9300
3,Chile,1641222,39,2024-01-01,2026-08-12,1498518,97625,42.8300
4,Colombia,192,1,2025-08-06,2026-04-30,192,0,0.0000
5,United States,64,1,2026-03-02,2026-06-26,64,0,0.0000


In [ ]:
# Distribución por JOB_CLASSIFICATION en Brasil
# Pregunta crítica: ¿el filtro Rep de Envio 1+2 deja fuera algo relevante?
df_vol_job = bq_query(f"""
    SELECT JOB_CLASSIFICATION,
        COUNT(*)                                               AS registros,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2)          AS pct_del_total,
        ROUND(SUM(DOTACION),0)                                 AS dotacion_total,
        ROUND(SUM(ABSENCE_MANAGEABLE),0)                      AS abs_gestionable_total
    FROM `{TBL_ABS}`
    WHERE COUNTRY='{PAIS}' AND YEAR_MONTH_DAY>=DATE '{FECHA_INICIO}'
      AND ABSENCE_TYPE IN ('Dotacion','Gestionable')
    GROUP BY JOB_CLASSIFICATION ORDER BY registros DESC
""")
print(f'Clasificaciones encontradas en Brasil: {len(df_vol_job)}')
display(df_vol_job)

pct = df_vol_job[df_vol_job['JOB_CLASSIFICATION'].isin(['Rep de Envio 1','Rep de Envio 2'])]['pct_del_total'].sum()
print(f'\nFiltro actual (Rep de Envio 1+2) cubre {pct:.1f}% del total Brasil')


In [ ]:
# Distribución por ABSENCE_TYPE y REASON_ABSENCE (Brasil, filtro TTE)
df_vol_abs = bq_query(f"""
    SELECT ABSENCE_TYPE, UPPER(REASON_ABSENCE) AS REASON_ABSENCE,
        COUNT(*)                                              AS registros,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2)         AS pct,
        ROUND(SUM(DOTACION),0)                                AS dotacion_sum,
        ROUND(SUM(ABSENCE_MANAGEABLE),0)                     AS abs_gest_sum
    FROM `{TBL_ABS}`
    WHERE COUNTRY='{PAIS}' AND YEAR_MONTH_DAY>=DATE '{FECHA_INICIO}'
      AND JOB_CLASSIFICATION IN ({JOB_SQL})
    GROUP BY ABSENCE_TYPE, UPPER(REASON_ABSENCE)
    ORDER BY registros DESC
""")
print('Distribucion por ABSENCE_TYPE y REASON_ABSENCE - Brasil TTE')
display(df_vol_abs)


In [ ]:
# Sitios únicos — con y sin match en el clasificador REG
df_vol_sites = bq_query(f"""
    SELECT AB.SITE,
        IFNULL(REG.nome_atual,'(sin match)') AS Operational_name,
        IFNULL(REG.logistica, '(sin match)') AS logistica,
        IFNULL(REG.Site,      '(sin match)') AS Tipo_OPS,
        COUNT(*)                              AS registros,
        MIN(AB.YEAR_MONTH_DAY)               AS desde,
        MAX(AB.YEAR_MONTH_DAY)               AS hasta
    FROM `{TBL_ABS}` AB
    LEFT JOIN `{TBL_REG}` REG ON UPPER(AB.SITE)=UPPER(REG.Ubicacion__Nombre)
    WHERE AB.COUNTRY='{PAIS}' AND AB.YEAR_MONTH_DAY>=DATE '{FECHA_INICIO}'
      AND AB.JOB_CLASSIFICATION IN ({JOB_SQL})
      AND AB.ABSENCE_TYPE IN ('Dotacion','Gestionable')
    GROUP BY ALL ORDER BY registros DESC
""")
sin_match = (df_vol_sites['Operational_name']=='(sin match)').sum()
print(f'Sitios unicos: {df_vol_sites["SITE"].nunique()} | Sin match en REG: {sin_match}')
display(df_vol_sites)


---
## Celda 5 — Exploración de la Tabla Madre Turnover (TO)

In [ ]:
df_schema_to = bq_query("""
    SELECT column_name, data_type, is_nullable
    FROM `meli-people.SILVER_PE_SHIPPING.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name='KPI_LATAM_NC_TO_ALL'
    ORDER BY ordinal_position
""")
print(f'Columnas tabla madre TO: {len(df_schema_to)}')
display(df_schema_to)


In [ ]:
df_vol_to = bq_query(f"""
    SELECT Pais_Region,
        COUNT(*)                                                AS registros,
        COUNT(DISTINCT Ubicacion__Nombre)                       AS sitios_unicos,
        MIN(Ano) AS ano_min, MAX(Ano) AS ano_max,
        COUNTIF(tipoBaja IN ('Renuncia','Despido','Abandono de empleo')) AS total_bajas
    FROM `{TBL_TO}`
    WHERE Ano>=2024
    GROUP BY Pais_Region ORDER BY registros DESC LIMIT 20
""")
print('Distribucion por Pais/Region - tabla madre TO')
display(df_vol_to)


In [ ]:
df_agrup_to = bq_query(f"""
    SELECT Agrupador_1, Clasificacion_de_puestos_Descripcion,
        COUNT(*) AS registros,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2) AS pct
    FROM `{TBL_TO}`
    WHERE Pais_Region='{PAIS}' AND Ano>=2024
    GROUP BY Agrupador_1, Clasificacion_de_puestos_Descripcion
    ORDER BY registros DESC LIMIT 40
""")
print('Agrupadores TO - Brasil 2024+')
display(df_agrup_to)


In [ ]:
df_tipobaja = bq_query(f"""
    SELECT tipoBaja, Motivo_del_evento, COUNT(*) AS registros,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2) AS pct
    FROM `{TBL_TO}`
    WHERE Pais_Region='{PAIS}' AND Ano>=2024
      AND Agrupador_1 IN ('Representante','Representante eventual',
          'Representantes - CDBR','Representante Inicial',
          'Rep eventual - Peak','Representante Externo')
      AND Clasificacion_de_puestos_Descripcion IN ({JOB_SQL})
    GROUP BY tipoBaja, Motivo_del_evento ORDER BY registros DESC
""")
print('Tipos de Baja - Brasil TTE 2024+')
display(df_tipobaja)


---
## Celda 6 — Nuestra Query de Agregación ABS

Esta es **nuestra versión propia** del dataset de trabajo, construida directo desde Silver.
No dependemos de ninguna tabla DW de terceros.

**Mejoras vs el SQL de referencia:**
- `TIPO_CONTRATO = 'DESCONOCIDO'` en vez de espacio vacío (trazabilidad)
- `logistica` se mantiene como columna extra para auditoría
- `JOB_CLASSIFICATION` en el output para posibles splits futuros
- Cache en Parquet: solo golpea BQ cuando hay datos nuevos


In [ ]:
CACHE_ABS = DATA_DIR / 'abs_agregado_raw.parquet'


def pull_abs_from_bq():
    """Query directa a Silver. Equivale al DW pero sin necesitar permisos de CREATE."""
    q = f"""
    SELECT
        EXTRACT(YEAR  FROM AB.YEAR_MONTH_DAY) AS ANO,
        EXTRACT(MONTH FROM AB.YEAR_MONTH_DAY) AS MES,
        AB.SITE,
        IFNULL(REG.nome_atual,'(sin clasificador)') AS Operational_name,
        IFNULL(REG.Site,      '(sin clasificador)') AS Tipo_OPS,
        IFNULL(REG.logistica, '(sin clasificador)') AS logistica,
        AB.ABSENCE_TYPE,
        UPPER(AB.REASON_ABSENCE)                     AS REASON_ABSENCE,
        AB.QUALIFIER_CODE,
        CASE
            WHEN UPPER(AB.REASON_ABSENCE)='DOTACION'              THEN 'a.Dotacion'
            WHEN UPPER(AB.REASON_ABSENCE)='AUSENCIA INJUSTIFICADA' THEN 'b.Falta_Inj'
            WHEN UPPER(AB.REASON_ABSENCE)='ENFERMEDAD'            THEN 'c.Atestados_Medicos'
            ELSE 'd.rest_abs_gestionable'
        END AS TIPO_ABS_GESTIONABLE,
        CASE
            WHEN AB.EXTERNAL_CONTRACTOR='No' THEN 'MELI'
            WHEN AB.EXTERNAL_CONTRACTOR='Si' THEN 'EXTERNO'
            ELSE 'DESCONOCIDO'
        END AS TIPO_CONTRATO,
        AB.JOB_CLASSIFICATION,
        SUM(AB.DOTACION)            AS Dotacion_programada,
        SUM(AB.ABSENCE_MANAGEABLE)  AS Ausentismo_gestionable
    FROM `{TBL_ABS}` AB
    LEFT JOIN `{TBL_REG}` REG ON UPPER(AB.SITE)=UPPER(REG.Ubicacion__Nombre)
    WHERE AB.COUNTRY='{PAIS}'
      AND AB.YEAR_MONTH_DAY >= DATE '{FECHA_INICIO}'
      AND AB.ABSENCE_TYPE IN ('Dotacion','Gestionable')
      AND AB.JOB_CLASSIFICATION IN ({JOB_SQL})
      AND IFNULL(REG.logistica,'MELILOG') IN ('MELILOG','3PL/MELILOG')
      AND (REG.nome_atual IS NULL OR REG.nome_atual NOT LIKE 'FBM%')
    GROUP BY ALL
    ORDER BY ANO, MES, Operational_name
    """
    print('Ejecutando query ABS en BigQuery...')
    df = bq_query(q)
    df['fecha_mes'] = pd.to_datetime(
        df['ANO'].astype(str) + '-' + df['MES'].astype(str).str.zfill(2) + '-01'
    )
    return df


def cargar_abs(force_refresh=False):
    if not force_refresh and CACHE_ABS.exists():
        df = pd.read_parquet(CACHE_ABS)
        ultimo     = df['fecha_mes'].max()
        mes_ant    = pd.Timestamp.now().replace(day=1) - pd.offsets.MonthBegin(1)
        if ultimo >= mes_ant:
            print(f'OK ABS desde cache (ultimo mes: {ultimo.strftime("%Y-%m")})')
            return df
        print(f'Cache desactualizado ({ultimo.strftime("%Y-%m")}). Re-pulling...')
    df = pull_abs_from_bq()
    df.to_parquet(CACHE_ABS)
    print(f'OK ABS guardado en cache: {CACHE_ABS}')
    return df


df_abs_raw = cargar_abs(force_refresh=False)
print(f'Shape  : {df_abs_raw.shape}')
print(f'Rango  : {df_abs_raw["fecha_mes"].min().strftime("%Y-%m")} a {df_abs_raw["fecha_mes"].max().strftime("%Y-%m")}')
print(f'Sitios : {df_abs_raw["Operational_name"].nunique()}')
display(df_abs_raw.head(10))


---
## Celda 7 — Auditoría de Calidad de Datos

**Qué buscamos:**
- Nulos por columna
- Valores imposibles (Dotacion <= 0, ausentismo negativo, ratio > 100%)
- Duplicados en el grano esperado (sitio × mes × tipo)
- Sitios sin match en el clasificador REG
- Gaps temporales por sitio (meses faltantes)

Cada hallazgo se documenta y la decisión de tratamiento se toma **explícitamente**.


In [ ]:
# Nulos por columna
def audit_nulos(df):
    nulos = df.isnull().sum()
    pct   = (nulos/len(df)*100).round(2)
    return pd.DataFrame({'nulos':nulos,'pct_%':pct}).query('nulos>0').sort_values('nulos',ascending=False)

dn = audit_nulos(df_abs_raw)
if len(dn):
    print('Columnas con nulos:')
    display(dn)
else:
    print('OK Sin valores nulos')


In [ ]:
# Valores imposibles
dot_neg = df_abs_raw[
    (df_abs_raw['TIPO_ABS_GESTIONABLE']=='a.Dotacion') & (df_abs_raw['Dotacion_programada']<=0)
]
print(f'Dotacion <= 0 en registros Dotacion: {len(dot_neg):,}')
if len(dot_neg): display(dot_neg.head())

abs_neg = df_abs_raw[df_abs_raw['Ausentismo_gestionable']<0]
print(f'Ausentismo_gestionable < 0: {len(abs_neg):,}')
if len(abs_neg): display(abs_neg.head())

abs_may = df_abs_raw[
    (df_abs_raw['Ausentismo_gestionable']>df_abs_raw['Dotacion_programada']) &
    (df_abs_raw['Dotacion_programada']>0)
]
print(f'Ausentismo > Dotacion (ratio > 100%): {len(abs_may):,}')
if len(abs_may): display(abs_may.head())


In [ ]:
# Duplicados en el grano esperado
grano = ['Operational_name','fecha_mes','TIPO_CONTRATO','TIPO_ABS_GESTIONABLE','JOB_CLASSIFICATION']
dupes = df_abs_raw[df_abs_raw.duplicated(subset=grano,keep=False)]
print(f'Duplicados en el grano: {len(dupes):,}')
if len(dupes): display(dupes.head(20))
else: print('OK Sin duplicados')


In [ ]:
# Gaps temporales por sitio
df_dot = df_abs_raw[
    df_abs_raw['TIPO_ABS_GESTIONABLE']=='a.Dotacion'
].groupby(['Operational_name','fecha_mes'])['Dotacion_programada'].sum().reset_index()

all_m  = pd.date_range(df_dot['fecha_mes'].min(), df_dot['fecha_mes'].max(), freq='MS')
idx    = pd.MultiIndex.from_product([df_dot['Operational_name'].unique(), all_m],
                                     names=['Operational_name','fecha_mes'])
df_gap = pd.DataFrame(index=idx).reset_index().merge(df_dot, on=['Operational_name','fecha_mes'], how='left')
df_gap['ok'] = df_gap['Dotacion_programada'].notna()

df_gs = df_gap.groupby('Operational_name').agg(
    meses_tot = ('fecha_mes','count'),
    meses_ok  = ('ok','sum'),
    pct_cob   = ('ok', lambda x: round(x.mean()*100,1))
).reset_index()
df_gs['meses_faltantes'] = df_gs['meses_tot'] - df_gs['meses_ok']
df_gs = df_gs.sort_values('pct_cob')

print(f'Total sitios: {len(df_gs)}')
print(f'  Cobertura 100%   : {(df_gs["pct_cob"]==100).sum()}')
print(f'  Con meses faltantes: {(df_gs["pct_cob"]<100).sum()}')
print(f'  Con <50% cobertura : {(df_gs["pct_cob"]<50).sum()} (candidatos a excluir)')
display(df_gs)


---
## Celda 8 — Análisis de Distribuciones

Pivoteamos a nivel sitio × mes y analizamos las distribuciones de las métricas objetivo:
`ABS_Total`, `ABS_Falta`, `ABS_Atestados`, `ABS_Rest`, `Dotacion`.

Entender la distribución es clave para elegir el modelo correcto y el tratamiento de outliers.


In [ ]:
# Pivot al nivel del modelo (sitio x mes, MELI+EXTERNO combinados)
df_pb = df_abs_raw.groupby(
    ['Operational_name','SITE','Tipo_OPS','fecha_mes','TIPO_ABS_GESTIONABLE']
).agg(dot=('Dotacion_programada','sum'), ab=('Ausentismo_gestionable','sum')).reset_index()

df_pb['v'] = np.where(df_pb['TIPO_ABS_GESTIONABLE']=='a.Dotacion', df_pb['dot'], df_pb['ab'])

df_pivot = df_pb.pivot_table(
    index=['Operational_name','SITE','Tipo_OPS','fecha_mes'],
    columns='TIPO_ABS_GESTIONABLE', values='v', aggfunc='sum'
).reset_index()
df_pivot.columns.name = None
df_pivot = df_pivot.rename(columns={
    'a.Dotacion':'Dotacion','b.Falta_Inj':'Falta_Inj',
    'c.Atestados_Medicos':'Atestados_Medicos','d.rest_abs_gestionable':'Rest_Abs_Gestionable'
})
for c in ['Dotacion','Falta_Inj','Atestados_Medicos','Rest_Abs_Gestionable']:
    if c not in df_pivot.columns:
        df_pivot[c] = 0.0
    df_pivot[c] = df_pivot[c].fillna(0)

df_pivot['Ausencia_Total'] = df_pivot['Falta_Inj'] + df_pivot['Atestados_Medicos'] + df_pivot['Rest_Abs_Gestionable']
m = df_pivot['Dotacion'] > 0
df_pivot['ABS_Total']    = np.where(m, df_pivot['Ausencia_Total']/df_pivot['Dotacion'], np.nan)
df_pivot['ABS_Falta']    = np.where(m, df_pivot['Falta_Inj']/df_pivot['Dotacion'], np.nan)
df_pivot['ABS_Atestado'] = np.where(m, df_pivot['Atestados_Medicos']/df_pivot['Dotacion'], np.nan)
df_pivot['ABS_Rest']     = np.where(m, df_pivot['Rest_Abs_Gestionable']/df_pivot['Dotacion'], np.nan)

print(f'df_pivot: {df_pivot.shape}')
print(f'Sitios: {df_pivot["Operational_name"].nunique()} | Meses: {df_pivot["fecha_mes"].nunique()}')
display(df_pivot.describe())


In [ ]:
mets    = ['ABS_Total','ABS_Falta','ABS_Atestado','ABS_Rest','Dotacion']
cols    = ['#2980b9','#e74c3c','#27ae60','#f39c12','#8e44ad']
fig, ax = plt.subplots(2, 3, figsize=(18,10))
ax      = ax.flatten()
rows    = []

for i,(met,color) in enumerate(zip(mets,cols)):
    d = df_pivot[met].dropna()
    d = d[d>0]
    ax[i].hist(d, bins=50, color=color, alpha=0.75, edgecolor='white')
    ax[i].axvline(d.mean(),   color='black', ls='--', lw=1.5, label=f'Media={d.mean():.4f}')
    ax[i].axvline(d.median(), color='gray',  ls=':',  lw=1.5, label=f'Mediana={d.median():.4f}')
    ax[i].set_title(met, fontsize=13, fontweight='bold')
    ax[i].legend(fontsize=8)
    ax[i].text(0.97,0.97,f'n={len(d):,}\np99={d.quantile(.99):.4f}\nSkew={d.skew():.2f}',
               transform=ax[i].transAxes, ha='right', va='top', fontsize=8,
               bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.5))
    rows.append({'Metrica':met,'n':len(d),'Media':d.mean(),'Mediana':d.median(),
                 'Std':d.std(),'p5':d.quantile(.05),'p25':d.quantile(.25),
                 'p75':d.quantile(.75),'p95':d.quantile(.95),'p99':d.quantile(.99),
                 'Skewness':d.skew(),'Kurtosis':d.kurtosis()})

ax[-1].set_visible(False)
plt.suptitle('Distribuciones - TTE Brasil (sitio x mes, ceros excluidos)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('\nEstadisticas:')
display(pd.DataFrame(rows).set_index('Metrica').round(4))


---
## Celda 9 — Detección y Tratamiento de Outliers

**Tres enfoques complementarios:**
1. **IQR (Tukey)** — robusto ante distribuciones asimétricas
2. **Z-score modificado (MAD)** — más robusto que el Z clásico ante heavy tails
3. **Boxplot interactivo** — para distinguir errores de picos estacionales reales

> En series de tiempo, un outlier puede ser real (brote de gripe, huelga, evento climático).  
> La decisión de winsorizar, marcar o eliminar se hace **explícitamente**, no de forma ciega.


In [ ]:
def out_iqr(s, f=1.5):
    Q1,Q3 = s.quantile([.25,.75])
    return (s<Q1-f*(Q3-Q1)) | (s>Q3+f*(Q3-Q1))


def out_zmod(s, u=3.5):
    med = s.median()
    mad = (s-med).abs().median()
    if mad==0: return pd.Series(False, index=s.index)
    return (0.6745*(s-med)/mad).abs() > u


df_out = df_pivot[df_pivot['Dotacion']>0].copy()
for met in ['ABS_Total','ABS_Falta','ABS_Atestado','ABS_Rest']:
    s = df_out[met].fillna(0)
    df_out[f'{met}_oi'] = out_iqr(s)
    df_out[f'{met}_oz'] = out_zmod(s)
    df_out[f'{met}_ob'] = df_out[f'{met}_oi'] & df_out[f'{met}_oz']

rows = []
for met in ['ABS_Total','ABS_Falta','ABS_Atestado','ABS_Rest']:
    rows.append({'Metrica':met,
                 'IQR':df_out[f'{met}_oi'].sum(),
                 'Z-mod':df_out[f'{met}_oz'].sum(),
                 'Ambos (alta confianza)':df_out[f'{met}_ob'].sum()})
print('Resumen outliers (solo registros con Dotacion > 0):')
display(pd.DataFrame(rows).set_index('Metrica'))


In [ ]:
# Boxplot interactivo Top 30 sitios por mediana ABS_Total
top30 = df_out.groupby('Operational_name')['ABS_Total'].median().sort_values(ascending=False).head(30).index.tolist()
fig = px.box(
    df_out[df_out['Operational_name'].isin(top30)],
    x='Operational_name', y='ABS_Total',
    category_orders={'Operational_name':top30},
    title='ABS_Total por Sitio - Top 30 por mediana (outliers como puntos)',
    color='Operational_name', points='outliers'
)
fig.update_layout(height=550, showlegend=False, xaxis_tickangle=-60, xaxis_title='')
fig.show()


In [ ]:
# Tabla de outliers de alta confianza para revisión manual
cols_s = ['Operational_name','fecha_mes','Dotacion',
           'Falta_Inj','Atestados_Medicos','Rest_Abs_Gestionable',
           'ABS_Total','ABS_Falta','ABS_Atestado','ABS_Rest']
df_ohc = df_out[df_out['ABS_Total_ob']][[c for c in cols_s if c in df_out]].sort_values('ABS_Total',ascending=False)
print(f'Outliers alta confianza: {len(df_ohc):,} registros en {df_ohc["Operational_name"].nunique()} sitios')
print('Revisar caso por caso antes de decidir tratamiento.')
display(df_ohc)


---
## Celda 10 — Análisis de Series Temporales por Sitio

**Estructura temporal antes de modelar:**
- ¿Los sitios muestran tendencia, estacionalidad o solo ruido?
- ¿Hay sitios con distintas fechas de inicio/fin (lifecycles)?
- ¿Cuántos meses reales tiene cada sitio?

Esto define el mínimo de historia requerido y la estrategia de feature engineering.


In [ ]:
df_hist = df_pivot[df_pivot['Dotacion']>0].groupby('Operational_name').agg(
    fecha_inicio = ('fecha_mes','min'),
    fecha_fin    = ('fecha_mes','max'),
    n_meses      = ('fecha_mes','count'),
    dot_media    = ('Dotacion','mean'),
    abs_media    = ('ABS_Total','mean'),
    abs_std      = ('ABS_Total','std')
).reset_index()
df_hist['cv_abs'] = (df_hist['abs_std']/df_hist['abs_media']).round(3)
df_hist = df_hist.sort_values('n_meses',ascending=False)

print('Historia por sitio:')
print(f'  >= 18 meses: {(df_hist["n_meses"]>=18).sum()}')
print(f'  >= 12 meses: {(df_hist["n_meses"]>=12).sum()}')
print(f'  >=  6 meses: {(df_hist["n_meses"]>=6).sum()}')
print(f'   <  6 meses: {(df_hist["n_meses"]<6).sum()} (candidatos a excluir)')
display(df_hist)


In [ ]:
# Heatmap: ABS_Total por Sitio x Mes
sitios_ok = df_hist[df_hist['n_meses']>=6]['Operational_name'].tolist()
df_hm = df_pivot[
    df_pivot['Operational_name'].isin(sitios_ok) & (df_pivot['Dotacion']>0)
].pivot_table(index='Operational_name', columns='fecha_mes', values='ABS_Total')

fig, ax = plt.subplots(figsize=(18, max(8, len(sitios_ok)*0.28)))
sns.heatmap(df_hm, ax=ax, cmap='RdYlGn_r', annot=False, linewidths=0.3,
            vmin=0, vmax=df_hm.stack().quantile(0.95),
            cbar_kws={'label':'ABS Total (ratio)'})
ax.set_title('Heatmap ABS_Total - Sitio x Mes (rojo=alto ausentismo, blanco=sin dato)', fontsize=13)
ax.set_xticklabels(
    [pd.Timestamp(t.get_text()).strftime('%b %y') for t in ax.get_xticklabels()],
    rotation=45, ha='right', fontsize=7
)
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()
print(f'Meses: {df_hm.shape[1]} | Sitios: {df_hm.shape[0]}')


In [ ]:
# Series temporales Top 6 sitios por dotacion media
top6 = df_pivot[df_pivot['Dotacion']>0].groupby('Operational_name')['Dotacion'].mean().nlargest(6).index.tolist()
fig  = make_subplots(rows=3, cols=2, subplot_titles=top6, vertical_spacing=0.12)
for i,site in enumerate(top6):
    r,c = divmod(i,2)
    ds  = df_pivot[df_pivot['Operational_name']==site].sort_values('fecha_mes')
    fig.add_trace(
        go.Scatter(x=ds['fecha_mes'], y=(ds['ABS_Total']*100).round(2),
                   mode='lines+markers', name='ABS%',
                   line=dict(color='#e74c3c',width=2), marker=dict(size=5)),
        row=r+1, col=c+1
    )
fig.update_layout(height=800, showlegend=False, template='plotly_white',
                  title_text='Series ABS_Total - Top 6 sitios por Dotacion media')
fig.update_yaxes(title_text='ABS %')
fig.show()


---
## Hallazgos de esta fase (completar al ejecutar)

| # | Hallazgo | Decisión |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

---

## Próximos notebooks

| Notebook | Qué haremos |
|---|---|
| **v1 — Limpieza** | Tratamiento de outliers, gaps, sitios a excluir |
| **v2 — Feature Engineering** | Lags, rolling, estacionalidad, covariables externas |
| **v3 — Modelado ABS** | Benchmark: Naive, SARIMA, CatBoost/LGBM, modelos fundacionales |
| **v4 — Modelado TO** | Mismo proceso para Turnover |
| **v5 — Validación** | Walk-forward riguroso, métricas por sitio |
| **v6 — Producción** | Pipeline completo, schedule mensual |
